**Step-1:**
Loading the dataset before executing the queries to ensure outputs are generated.

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, sum, min, max, mean
from pyspark.sql.types import TimestampType

# Initialize session and ingest the target dataset
spark = SparkSession.builder.appName("Week5_Sales_ETL").getOrCreate()
df_sales = spark.read.csv("week5_sales_dataset.csv", header=True, inferSchema=True)

df_sales.show(5)

+--------+----------+------------+----------------+-------+----------------+------+-------------+---+------------+-----------------+--------+-------------------+-----------+-------+--------+
|store_id|product_id|product_name|product_category|user_id|transaction_date|region|         city|age|subscription|            email|username|      raw_timestamp|sale_amount|  price|  status|
+--------+----------+------------+----------------+-------+----------------+------+-------------+---+------------+-----------------+--------+-------------------+-----------+-------+--------+
|    S004|      P125|    Mousepad|     Accessories|  U0681|      2023-12-14| North|      Chicago| 28|     Premium|user681@email.com| user681|2023-12-14 10:03:00|      14.43|  13.18|  Active|
|    S002|      P121|         Pen|      Stationery|  U0732|      2023-07-19| North|      Detroit| 46|       Basic|user732@email.com| user732|2023-07-19 14:46:00|       3.03|   3.28|Inactive|
|    S002|      P101|      Laptop|     Electr

**Implementation Logic:**
Deduplication on specific primary keys prevents double-counting transactions, ensuring financial reporting accuracy.

In [15]:
df_deduped = df_sales.dropDuplicates(["user_id", "transaction_date"])

print(f"Original Count: {df_sales.count()} | Deduplicated Count: {df_deduped.count()}")
df_deduped.show(5)

Original Count: 1020 | Deduplicated Count: 970
+--------+----------+-------------+----------------+-------+----------------+------+-----------+---+------------+---------------+--------+-------------------+-----------+------+--------+
|store_id|product_id| product_name|product_category|user_id|transaction_date|region|       city|age|subscription|          email|username|      raw_timestamp|sale_amount| price|  status|
+--------+----------+-------------+----------------+-------+----------------+------+-----------+---+------------+---------------+--------+-------------------+-----------+------+--------+
|    S001|      P108|   Headphones|     Electronics|  U0001|      2023-05-21|  West|Los Angeles| 33|     Premium|user1@email.com|   user1|2023-05-21 19:57:00|     123.14|  NULL|  Active|
|    S005|      P133|       Pillow|       Furniture|  U0002|      2023-01-14|  West|Los Angeles| 53|     Premium|user2@email.com|   user2|2023-01-14 14:14:00|       44.3| 42.59|  Active|
|    S003|      P1

**Implementation Logic:** Utilizing modular method chaining to filter data before grouping limits the amount of data that needs to be shuffled across the network, highly optimizing query performance.

In [16]:
df_west_avg_sales = (df_sales
   .filter(col("region") == "West")
   .groupBy("product_category")
   .agg(avg("sale_amount").alias("avg_sales"))
)

df_west_avg_sales.show()

+----------------+------------------+
|product_category|         avg_sales|
+----------------+------------------+
|      Stationery|15.570845070422532|
|     Electronics|251.61533333333338|
|       Furniture| 345.8691139240506|
|     Accessories|28.144615384615374|
|      Appliances| 62.51857142857143|
+----------------+------------------+



In [17]:
df_filled_status = df_sales.na.fill("Unknown", subset=["status"])

# Displaying specific columns to prove the nulls were filled
df_filled_status.select("user_id", "status").show(5)

+-------+--------+
|user_id|  status|
+-------+--------+
|  U0681|  Active|
|  U0732|Inactive|
|  U0854| Unknown|
|  U0802|Inactive|
|  U0288|Inactive|
+-------+--------+
only showing top 5 rows


**Implementation Logic:** Applying a .filter() after the aggregation acts identically to a SQL HAVING clause, dynamically isolating high-volume dimensions (like our mocked Chicago data).

In [18]:
df_top_cities = (df_sales
    .groupBy("city")
    .count()
    .filter(col("count") > 100)
)

df_top_cities.show()

+-------------+-----+
|         city|count|
+-------------+-----+
|  Los Angeles|  198|
|San Francisco|  160|
|      Chicago|  145|
|     New York|  167|
+-------------+-----+



**Implementation Logic:** Wrapping multiple conditions in parentheses and using the bitwise & operator ensures Spark evaluates the complex boolean logic correctly. `.between()` improves code readability over `>=` and `<=`

In [19]:
df_target_demographic = df_sales.filter(
    (col("age").between(18, 30)) &
    (col("subscription") == "Premium")
)

df_target_demographic.select("user_id", "age", "subscription").show(5)

+-------+---+------------+
|user_id|age|subscription|
+-------+---+------------+
|  U0681| 28|     Premium|
|  U0798| 28|     Premium|
|  U0072| 22|     Premium|
|  U0682| 25|     Premium|
|  U0896| 25|     Premium|
+-------+---+------------+
only showing top 5 rows


**Implementation Logic:** Combining `.withColumn()` with `.cast()` normalizes the data type for time-series analysis, while dropping the legacy column optimizes the memory footprint of the DataFrame.

In [20]:
df_standardized_time = (df_sales
   .withColumn("event_time", col("raw_timestamp").cast(TimestampType()))
   .drop("raw_timestamp")
)

df_standardized_time.select("event_time").printSchema()
df_standardized_time.select("event_time").show(5)

root
 |-- event_time: timestamp (nullable = true)

+-------------------+
|         event_time|
+-------------------+
|2023-12-14 10:03:00|
|2023-07-19 14:46:00|
|2023-03-03 15:47:00|
|2023-05-22 13:53:00|
|2023-09-16 15:20:00|
+-------------------+
only showing top 5 rows


**Implementation Logic:** To remove the bad rows, we structure our filter to keep the good rows. We ensure high data quality by checking for both structural nulls `(isNotNull())` and empty strings `(!= "")`

In [22]:
df_clean_users = df_sales.filter(
   col("email").isNotNull() &
   (col("username") != "")
)

df_clean_users.select("user_id", "email", "username").show(10)

+-------+-----------------+--------+
|user_id|            email|username|
+-------+-----------------+--------+
|  U0681|user681@email.com| user681|
|  U0732|user732@email.com| user732|
|  U0854|user854@email.com| user854|
|  U0802|user802@email.com| user802|
|  U0288|user288@email.com| user288|
|  U0242|user242@email.com| user242|
|  U0602|user602@email.com| user602|
|  U0327|user327@email.com| user327|
|  U0038| user38@email.com|  user38|
|  U0324|user324@email.com| user324|
+-------+-----------------+--------+
only showing top 10 rows


**Implementation Logic**: Passing multiple functions into a single `.agg()` block prevents the need to calculate identical groupings multiple times. Descriptive aliases are applied immediately for clear schema definition.

In [23]:
df_price_statistics = df_sales.agg(
   min("price").alias("min_price"),
   max("price").alias("max_price"),
   mean("price").alias("mean_price")
)

df_price_statistics.show()

+---------+---------+------------------+
|min_price|max_price|        mean_price|
+---------+---------+------------------+
|     1.72|  1336.84|207.88046364594297|
+---------+---------+------------------+



**Implementation Logic:** This represents a production-grade ETL pipeline. By chaining the methods, we create a highly readable execution plan. Deduplication and null-handling occur before the aggregation to guarantee strict revenue accuracy.

In [24]:
df_revenue_pipeline = (df_sales
    .dropDuplicates()
    .na.fill(0, subset=["price"])
    .groupBy("store_id")
    .agg(sum("price").alias("total_revenue"))
)

df_revenue_pipeline.show()

+--------+------------------+
|store_id|     total_revenue|
+--------+------------------+
|    S004|35417.530000000006|
|    S001| 40919.27999999999|
|    S002|          35639.14|
|    S005|30323.860000000015|
|    S003| 42864.19000000002|
+--------+------------------+

